# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [mlcroissant](https://mlcommons.github.io/croissant/) library, following the Croissant data packaging standard for reproducible, FAIR machine learning datasets.

### Dataset Source
The dataset schema is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install mlcroissant if needed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview

Review available record sets, field IDs, and metadata. All entities are referenced by their `@id` field for clarity and reproducibility.


In [ ]:
# List all available record sets and their @id
print("Available Record Sets in the dataset:")
record_sets = dataset.record_sets
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. @id: {rs['@id']}")
    print(f"   Name: {rs.get('name', 'N/A')}")
    print(f"   Description: {rs.get('description', 'N/A')}")
    # List field @id and (name)
    if 'field' in rs:
        print("   Fields:")
        for field in rs['field']:
            if isinstance(field, dict):
                field_id = field.get('@id', '')
                field_name = field.get('name', '')
            else:
                field_id = field
                field_name = ''
            print(f"     - @id: {field_id} {('(name: '+field_name+')' if field_name else '')}")
    print()

## 3. Data Extraction

Load data from record sets into Pandas DataFrames. Use each record set and field `@id` as seen in the previous overview.

In [ ]:
# Reference record sets by their @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Map each record set @id to a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"Record set {record_set_id} is empty or not directly loadable.")

# For demonstration, select the first non-empty record set
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"\nFirst rows from record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with data available.")

## 4. Exploratory Data Analysis (EDA)

Apply exploratory steps: Filtering, normalization of numeric fields, and grouping by categorical fields by referencing them using their `@id`.

> _**Note:** If there are no numeric fields, you may adapt the analysis accordingly._

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    
    # Identify likely numeric fields using dtype or known @id
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].quantile(0.9)  # Use the 90th percentile as example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (90th percentile):")
        display(filtered_df.head())

        # Normalization
        norm_col = numeric_field_id + "_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Find a candidate group field (categorical, not numeric)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])):
                n_unique = df[col].nunique()
                if 1 < n_unique < 20:
                    group_field_id = col
                    break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields found in main record set. Please adjust analysis as needed.")
else:
    print("No record set DataFrame loaded for EDA.")

## 5. Visualization

Visualize distributions and relationships in the dataset using Pandas built-in plotting and Matplotlib. Again, use field `@id`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10, 6))
        df.groupby(group_field_id)[numeric_field_id].mean().sort_values().plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric or group field to visualize.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, inspect, and analyze a dataset defined by a Croissant schema with `mlcroissant`. We:
- Loaded the dataset and displayed metadata including the title and description
- Listed available record sets and fields by their `@id`
- Loaded records from each record set into Pandas DataFrames
- Applied exploratory data analysis: filtering and normalization of a numeric field, and grouping by a categorical field
- Visualized field distributions

For more in-depth analyses, consider reviewing the dataset documentation and experimenting with additional cleaning or modeling steps using the referenced field `@id`s. Always refer to the Croissant schema for detailed, machine-actionable dataset structure.